<a href="https://colab.research.google.com/github/johirbuet/cudapractise/blob/main/Cuda_Practise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUDA Practise




Practise CUDA


## Verify CUDA is enabled

* Run !nvidia-smi to verify GPU runtime is enabled
* Run !nvcc --version to verify CUDA compiler is running

In [12]:
!nvidia-smi

Thu May  7 04:58:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


# Step 1 — “Hello Kernel” + Thread Indexing
## ✅ Goal
By the end of this step you will be able to explain and use:

* What a kernel is (__global__) and that it runs on the GPU.
* How to launch a kernel with <<<numBlocks, threadsPerBlock>>>.
* How each thread identifies itself using blockIdx, blockDim, and threadIdx.

In [13]:
%%writefile hello.cu
#include <cstdio>
#include <cuda_runtime.h>

__global__ void hello_kernel() {
    int tid = (int)threadIdx.x;
    int bid = (int)blockIdx.x;
    int bdim = (int)blockDim.x;

    int global_id = bid * bdim + tid;
    printf("Hello from block %d, thread %d (global_id=%d)\n", bid, tid, global_id);
}

int main() {
    // Launch: 2 blocks, 4 threads per block
    hello_kernel<<<2, 4>>>();

    // Ensure kernel completes and surface any runtime errors
    cudaError_t err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        printf("CUDA error: %s\n", cudaGetErrorString(err));
        return 1;
    }

    return 0;
}

Overwriting hello.cu


### What’s happening here (quick intuition):

* hello_kernel is marked __global__, meaning it’s a kernel callable from host (CPU) and executed on device (GPU).
* The launch hello_kernel<<<2,4>>> creates a grid with 2 blocks and 4 threads per block = 8 threads total.
* Each thread computes global_id = blockIdx.x * blockDim.x + threadIdx.x.

In [17]:
# Compile
!nvcc hello.cu -o hello

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [18]:
# Run
!./hello

Hello from block 1, thread 0 (global_id=4)
Hello from block 1, thread 1 (global_id=5)
Hello from block 1, thread 2 (global_id=6)
Hello from block 1, thread 3 (global_id=7)
Hello from block 0, thread 0 (global_id=0)
Hello from block 0, thread 1 (global_id=1)
Hello from block 0, thread 2 (global_id=2)
Hello from block 0, thread 3 (global_id=3)


# Step 2 — Vector Addition (device memory + copies + kernel)
This is the canonical first CUDA “real” program: each GPU thread computes one element of `C[i] = A[i] + B[i]`. NVIDIA even ships a vectorAdd sample demonstrating exactly this.

## ✅ What you’ll learn in Step 2

* Allocating GPU memory: cudaMalloc
* Copying data Host↔Device: cudaMemcpy
* Launching a kernel to cover N elements (grid sizing)
* Why kernels need bounds checks (if (i < N))

In [30]:
%%writefile vector_add.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do {                                  \
  cudaError_t err = call;                                      \
  if (err != cudaSuccess) {                                    \
    printf("CUDA error %s:%d: %s\n",                            \
           __FILE__, __LINE__, cudaGetErrorString(err));       \
    exit(1);                                                   \
  }                                                            \
} while(0)

__global__ void vec_add(const float* A, const float* B, float* C, int N) {
    int i = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    const int N = 1 << 20;                  // ~1 million elements
    const size_t bytes = (size_t)N * sizeof(float);

    // Host allocations
    float* hA = (float*)malloc(bytes);
    float* hB = (float*)malloc(bytes);
    float* hC = (float*)malloc(bytes);

    // Initialize input
    for (int i = 0; i < N; i++) {
        hA[i] = (float)i;
        hB[i] = 2.0f * (float)i;
    }

    // Device allocations
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc((void**)&dA, bytes));
    CUDA_CHECK(cudaMalloc((void**)&dB, bytes));
    CUDA_CHECK(cudaMalloc((void**)&dC, bytes));

    // Copy H -> D
    CUDA_CHECK(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));

    // Launch kernel
    int threadsPerBlock = 256;
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;
    vec_add<<<blocks, threadsPerBlock>>>(dA, dB, dC, N);

    // Catch launch/runtime errors and wait for completion
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    // Copy D -> H
    CUDA_CHECK(cudaMemcpy(hC, dC, bytes, cudaMemcpyDeviceToHost));

    // Verify a few values
    bool ok = true;
    for (int i : {0, 1, 2, 123, N-1}) {
        float expected = hA[i] + hB[i];
        if (hC[i] != expected) {
            printf("Mismatch at %d: got %f expected %f\n", i, hC[i], expected);
            ok = false;
            break;
        }
        printf("index = %d, value = %f\n", i, hC[i]);
    }
    printf(ok ? "OK ✅\\n" : "FAILED ❌\\n");

    // Cleanup
    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    free(hA); free(hB); free(hC);

    return ok ? 0 : 1;
}

Overwriting vector_add.cu


In [31]:
# Compile
!nvcc vector_add.cu -O2 -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [32]:
# Run
!./vector_add

index = 0, value = 0.000000
index = 1, value = 3.000000
index = 2, value = 6.000000
index = 123, value = 369.000000
index = 1048575, value = 3145725.000000
OK ✅\n

# What does cudaMemcpyHostToDevice mean here?
```
cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice)
```

### Answer:
It copies bytes of data from host (CPU) memory (hA) to device (GPU) memory (dA). This is the standard memory transfer step in CUDA runtime workflows

# If N = 1000 and threadsPerBlock = 256, what is blocks?
Formula:
```
blocks = (N + T - 1) / T
```
Compute:
```
blocks = (1000 + 256 - 1) / 256
blocks = 1255 / 256
Integer division ⇒ 4
```

### ✅ Answer: blocks = 4.

# Step 3 — Debugging & Correctness Habits (Error checks + Sync + “break it on purpose”)
## What you’ll learn in this step

1. Why CUDA kernel launches are asynchronous (CPU continues while GPU runs).
2. How to catch two classes of errors:
  * launch/configuration errors (caught by cudaGetLastError)
  * runtime execution errors (often revealed after cudaDeviceSynchronize)


3. Why bounds checks prevent illegal memory accesses in kernels.

In [38]:
%%writefile vector_add_buggy.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do {                                  \
  cudaError_t err = call;                                      \
  if (err != cudaSuccess) {                                    \
    printf("CUDA error %s:%d: %s\n",                            \
           __FILE__, __LINE__, cudaGetErrorString(err));       \
    exit(1);                                                   \
  }                                                            \
} while(0)

__global__ void vec_add_buggy(const float* A, const float* B, float* C, int N) {
    int i = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    // BUG: no bounds check
    C[i] = A[i] + B[i];
}

int main() {
    const int N = (1 << 20) + 13;
    const size_t bytes = (size_t)N * sizeof(float);

    float* hA = (float*)malloc(bytes);
    float* hB = (float*)malloc(bytes);
    float* hC = (float*)malloc(bytes);

    for (int i = 0; i < N; i++) {
        hA[i] = (float)i;
        hB[i] = 2.0f * (float)i;
    }

    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc((void**)&dA, bytes));
    CUDA_CHECK(cudaMalloc((void**)&dB, bytes));
    CUDA_CHECK(cudaMalloc((void**)&dC, bytes));

    CUDA_CHECK(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));

    int threadsPerBlock = 256;
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;
    vec_add_buggy<<<blocks, threadsPerBlock>>>(dA, dB, dC, N);

    // We still check for launch errors + synchronize to surface runtime errors.
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(hC, dC, bytes, cudaMemcpyDeviceToHost));

    printf("Done (buggy run)\n");

    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    free(hA); free(hB); free(hC);
    return 0;
}


Overwriting vector_add_buggy.cu


In [39]:
# Compile and Run
!nvcc vector_add_buggy.cu -O2 -o vector_add_buggy
!./vector_add_buggy

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Done (buggy run)


In [40]:
!compute-sanitizer --tool memcheck ./vector_add_buggy

========= COMPUTE-SANITIZER
========= Invalid __global__ read of size 4 bytes
=========     at vec_add_buggy(const float *, const float *, float *, int)+0x70
=========     by thread (32,0,0) in block (4096,0,0)
=========     Address 0x7fbded400080 is out of bounds
=========     and is 77 bytes after the nearest allocation at 0x7fbded000000 of size 4,194,356 bytes
=========     Saved host backtrace up to driver entry point at kernel launch time
=========         Host Frame: main [0x8b93] in vector_add_buggy
========= Invalid __global__ read of size 4 bytes
=========     at vec_add_buggy(const float *, const float *, float *, int)+0x70
=========     by thread (33,0,0) in block (4096,0,0)
=========     Address 0x7fbded400084 is out of bounds
=========     and is 81 bytes after the nearest allocation at 0x7fbded000000 of size 4,194,356 bytes
=========     Saved host backtrace up to driver entry point at kernel launch time
=========         Host Frame: main [0x8b93] in vector_add_buggy
====